# Beatles Project
### Extract scientific publications with Beatles titles

In [ ]:
# Standard library imports
import json
import random
import unicodedata
import re
import time
import pandas as pd
from typing import List, Tuple, Set

# Third-party imports
import bs4
import numpy as np
import requests
import urllib.parse
from tqdm.notebook import tqdm
from rapidfuzz import fuzz

# Local application imports
from dotenv import load_dotenv
import os
load_dotenv()
MY_API_KEY = os.environ["SCOPUS_API_KEY"]

In [ ]:
# Define constants
BATCH_SIZE = 25
MAX_RESULTS = 1_000

In [ ]:
def get_songs(fname: str) -> List[str]:
    songs = {}
    first = True
    with open(fname, "r", encoding="utf-8") as f_in:
        for line in f_in:
            if first:
                first = False
                continue
            split = line.strip().split("\t")
            song_name = split[0].strip('“”"')
            songs[song_name] = int(split[1])

    return songs

In [ ]:
def build_url(song_name: str) -> Tuple[str, str]:
    """Construct the Scopus API URL and encoded query based on song title."""

    song_name = re.sub(r"^And\b\s*", "And* ", song_name, flags=re.IGNORECASE)

    if "(" in song_name and ")" in song_name:
        split_name = song_name.split("(")
        pure_name = split_name[0].strip()
        additional = split_name[1][:-1].strip()
        query = (
            f"TITLE({pure_name}) OR TITLE({additional})"
        )
    else:
        query = f"TITLE({song_name})"
    
    if "." in song_name:
        without_periods = song_name.replace(".", "")
        query += (
            f" OR TITLE({without_periods})"
        )

    query_encoded = urllib.parse.quote(query, safe="")

    url = (
        "https://api.elsevier.com/content/search/scopus"
        f"?query={query_encoded}&count={BATCH_SIZE}"
    )

    return url, query_encoded

In [ ]:
def build_urls(song_name: str) -> List[Tuple[str, str, str]]:
    """Construct Scopus API URLs for all relevant fuzzy variants of a Beatles song title."""
    urls = []
    original = song_name.strip()

    # Step 1: Always remove article
    for article in ["The ", "A ", "An "]:
        if original.startswith(article):
            song_name = original[len(article):]
            break
    else:
        song_name = original

    # Step 2: Clean parentheses if present (split off optional subtitle or extra word)
    if "(" in song_name and ")" in song_name:
        split_name = song_name.split("(")
        base = split_name[0].strip()
        in_parens = split_name[1][:-1].strip()
        variants = [song_name, base, in_parens]
    else:
        variants = [song_name]

    # Step 3: Add dotless version if needed
    if "." in song_name:
        variants.append(song_name.replace(".", ""))

    # Step 6: Partial word-drop queries (1 word removed, excluding a/an/the, and excluding full match)
    words = song_name.split()
    ignore_words = {"a", "an", "the"}

    if len(words) > 2:
        for i, word in enumerate(words):
            if word.lower() in ignore_words:
                continue
            partial1 = [w for j, w in enumerate(words) if j < i]
            partial2 = [w for j, w in enumerate(words) if j > i]
            partial1_phrase = " ".join(partial1)
            partial2_phrase = " ".join(partial2)
            original_phrase = " ".join(words)
            if len(partial1) == 0:
                query = f'TITLE("{partial2_phrase}")'
            elif len(partial2) == 0:
                query = f'TITLE("{partial1_phrase}")'
            else:
                query = f'TITLE("{partial1_phrase}") AND TITLE("{partial2_phrase}")'
            query += f' AND NOT TITLE("{original_phrase}")'
            query_encoded = urllib.parse.quote(query, safe="")
            url = f"https://api.elsevier.com/content/search/scopus?query={query_encoded}&count={BATCH_SIZE}"
            urls.append((url, query, query_encoded))

        return urls
    else:
        return None

In [ ]:
song_name = "Love is All You Need"
ignore_words = {"a", "an", "the"}
words = song_name.split()
for i, word in enumerate(words):
        if word.lower() in ignore_words:
            continue
        partial1 = [w for j, w in enumerate(words) if j < i]
        partial2 = [w for j, w in enumerate(words) if j > i]
        partial1_phrase = " ".join(partial1)
        partial2_phrase = " ".join(partial2)
        original_phrase = " ".join(words)
        if len(partial1) == 0:
            query = f'TITLE("{partial2_phrase}")'
        elif len(partial2) == 0:
            query = f'TITLE("{partial1_phrase}")'
        else:
            query = f'TITLE("{partial1_phrase}") AND TITLE("{partial2_phrase}")'
        query += f' AND NOT TITLE("{original_phrase}")'
        print(query)


In [ ]:
test_list = ["The Long and Winding Road", "Hey Jude", "Let It Be", "Across the Universe", "Yesterday", "Lucy in the Sky with Diamonds"]
for song in test_list:
    tester = build_urls(song)
    if tester is not None:
        print(song)
        for url, query, query_encoded in tester:
            print(f"URL: {url}\nQuery: {query}\nEncoded Query: {query_encoded}\n")
        print("\n")

In [ ]:
replacements = {
    "æ": "ae", "ñ": "n", "ö": "oe", "ä": "ae", "ü": "ue", "ß": "ss",
    "ø": "o", "å": "a", "é": "e", "è": "e", "ç": "c", "ô": "o", "î": "i",
    "…": "...", "á": "a", "à": "a", "â": "a", "ã": "a", "ê": "e", "ë": "e",
    "í": "i", "ì": "i", "ï": "i", "ó": "o", "ò": "o", "õ": "o", "ú": "u",
    "ù": "u", "û": "u", "ý": "y", "ÿ": "y", "č": "c", "ž": "z", "š": "s",
    "ł": "l", "“": '"', "”": '"', "‘": "'", "’": "'", "–": "-", "—": "-",
    "·": "-", "°": " deg", "©": "", "®": "", "™": "", "µ": "u", "†": "*",
    "‡": "*", "×": "x"
}

In [ ]:
fields = {
    "scopus_id": "dc:identifier",
    "title": "dc:title",
    "cited_by": "citedby-count",
    "author": "dc:creator",
    "venue": "prism:publicationName",
    "doi": "prism:doi",
    "cover_date": "prism:coverDisplayDate",
}

headers = {"Accept": "application/json", "X-ELS-APIKey": MY_API_KEY}

In [ ]:
def extract_year_safe(date_str):
    # 1. Look for 4-digit year starting with 19 or 20
    match4 = re.search(r'\b(\d{2})\d{2}\b', date_str)
    if match4:
        return int(match4.group(0))
    
    # 2. Look for 2-digit number at the end
    match2 = re.search(r'(\d{2})\s*$', date_str)
    if match2:
        val = int(match2.group(1))
        if val <= 25:
            return 2000 + val
        elif 63 <= val <= 99:
            return 1900 + val
        else:
            return None
    
    return None

In [ ]:
def clean_line(line):
    global replacements

    # Normalize Unicode characters to closest ASCII equivalent
    for src, tgt in replacements.items():
        line = line.replace(src, tgt)
    line = unicodedata.normalize("NFKD", line)
    line = line.encode("ascii", "ignore").decode("ascii")
    
    return line

In [ ]:
def get_scopus_data(count: int, nr: int, song_name: str, song_year: int, fout: str) -> int:
    url_obj = build_urls(song)
    for url, query, query_encoded in url_obj:   
        #url, query = build_url(song_name)
        print(f"Query: {query}")
        cite_nr = 0

        resp = requests.get(url, headers=headers)

        if resp.status_code == 200:
            data = json.loads(resp.text.encode("utf-8"))
            if "search-results" in data:
                search_results = data["search-results"]

                wrote_results = False
                no_primary = search_results["opensearch:totalResults"]
                if no_primary.isdigit():
                    no_primary = int(no_primary)
                else:
                    no_primary = 0

                if no_primary > 0:
                    batch_nr = 0
                    while batch_nr * BATCH_SIZE < no_primary and cite_nr < MAX_RESULTS:
                        if "search-results" in data:
                            search_results = data["search-results"]
                            if no_primary > 0:
                                if "entry" in search_results:
                                    entries = search_results["entry"]
                                    if(len(entries) > 0):
                                        for entry in entries:
                                            scopus_id, title, cited_by, author, venue, doi, cover_date = (
                                                entry.get(key, "") for key in fields.values()
                                            )

                                            year = extract_year_safe(cover_date)
                                            if not type(year) == int or year < song_year:
                                                continue

                                            title = clean_line(title)
                                            author = clean_line(author)
                                            venue = clean_line(venue)

                                            fuzz_score = fuzz.partial_ratio(title.lower(), song_name.lower())

                                            with open(fout, "a", encoding="utf-8") as f_out:
                                                wrote_results = True
                                                f_out.write(
                                                        f"{count}\t{nr}\t{song_name}\t{cite_nr}\t{scopus_id}\t{title}\t{author}\t"
                                                        f"{cover_date}\t{year}\t{venue}\t{cited_by}\t{doi}\t{fuzz_score}\n"
                                                    )
                                            count += 1
                                            cite_nr += 1
                                            
                                    batch_nr += 1

                                    new_url = url + f"&start={batch_nr * BATCH_SIZE}"
                                    
                                    resp = requests.get(new_url, headers=headers)
                                    data = json.loads(resp.text.encode("utf-8"))
            else:
                print("No search results found.")

        else:
            print(f"Error: {resp.status_code}")
            print(resp.text)
            print(f"query: {query}")

    return count

In [ ]:
fname = "beatles_lyrics.txt"
fout = "20250715_scopus_lyrics_approx2.txt"

with open(fout, "w", encoding="utf-8") as f:
    f.write("row_nr\tsong_nr\tsong_name\tcite_nr\tscopus_id\ttitle\tauthor\tcover_date\tyear\tvenue\tcited_by\tdoi\tfuzz\n")

song_list = get_songs(fname=fname)

In [ ]:
#count, rejected = 0, 0
count = 0
for nr, song in enumerate(tqdm(song_list.keys())):
    if nr > -1:
        song_year = song_list[song]
        print(f"Song: {song}")
        count = get_scopus_data(count=count, nr=nr, song_name=song, song_year=song_year, fout=fout)
        time.sleep(1)

In [ ]:
print(nr)

"""""### OpenAI Prompting

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
MY_OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
MY_OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

MODEL = "gpt-4o-mini"
TEMPERATURE = 0.0

client = OpenAI(
    api_key=MY_OPENAI_API_KEY
)

In [ ]:
fname = "beatles_songs_wordplay.txt"
#fname = "beatles_albums.txt"
fin = "20250707_scopus_approx.txt"
#fin = "20250530_scopus_albums_approx.txt"
fout = "20250707_scopus_tags.txt"
#fout = "20250530_scopus_albums_tags.txt"

with open(fout, "w", encoding="utf-8") as f:
    f.write("scopus_id\ttitle\texplanation\twordplay\n")

song_list = get_songs(fname=fname)

In [ ]:
df = pd.read_csv(fin, sep="\t", encoding="utf-8")
print(df)

In [ ]:
# Below a dictionary with the token costs of gpt-4o-mini:
token_costs = {
    "gpt-4o-mini": {
        "input": 0.40,  # $1.10 per 1M input tokens
        "output": 1.60   # $4.40 per 1M output tokens
    }
}

In [ ]:
# List of known proper nouns/acronyms with casing to preserve
PROPER_TERMS = [
    "Canadian", "Psittacidae", "Dr.", "Google", "ChatGPT", "European", "Commission", "NFT",
    "DAC8", "ENT", "Br J Anaesth", "Web", "CD8+", "T cells", "IL-12/CD40L-mediated",
    "X", "Wii", "I", "Doull", "Foster", "Croatia", "PerformingGreenland", "Francis Underwood's", 
    "Tanizaki's The Key", "DCs", "STEMM", "Caryl Rusbult", "Irish", "Article 2B",
]

def apply_sentence_case(text: str) -> str:
    if not text:
        return text

    # Convert curly quotes ‘ ’ “ ” to straight quotes
    text = text.replace("‘", "`").replace("’", "'")
    text = text.replace("“", "``").replace("”", "''")

    # Convert straight quotes to LaTeX-style paired quotes
    text = re.sub(r'"([^"]+)"', r"``\1''", text)
    text = re.sub(r"'([^']+)'", r"`\1'", text)

    # Lowercase all text
    text = text.lower()

    # Capitalize first alphabetical character
    match = re.search(r"^([^\w]*)(\w)", text)
    if match:
        prefix, first_char = match.groups()
        text = prefix + first_char.upper() + text[match.end():]

    # Capitalize first letter after colon + optional whitespace
    text = re.sub(r'(:\s*)([a-z])', lambda m: m.group(1) + m.group(2).upper(), text)
    text = re.sub(r'(\?\s*)([a-z])', lambda m: m.group(1) + m.group(2).upper(), text)

    # Restore proper noun capitalization
    for term in sorted(PROPER_TERMS, key=len, reverse=True):
        escaped = re.escape(term.lower())
        pattern = r'\b' + escaped + r'\b'
        text = re.sub(pattern, term, text)

    return text

def generate_latex_table_body_to_file(sorted_df: pd.DataFrame, filename: str = "20250528_scopus_wordplay_table_full.tex"):
    """
    Generates LaTeX table body from sorted_df and writes it to a .tex file.
    Only shows song_name in the first row of each group and formats titles.
    """
    lines = []
    previous_song = None

    for _, row in sorted_df.iterrows():
        song_name = row["song_name"]
        title = row["title"]
        formatted_title = "-- " + apply_sentence_case(title.strip())

        if song_name != previous_song:
            song_cell = f"\\textbf{{{song_name}}}"
            previous_song = song_name
        else:
            song_cell = ""

        lines.append(f"{song_cell} & {formatted_title} \\\\")

    with open(filename, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

In [ ]:
generate_latex_table_body_to_file(sorted_df)

In [ ]:
print(len(aggregated_df[aggregated_df['manual'] == 'Yes']))
print(f"{len(aggregated_df[aggregated_df['manual'] == 'Yes'])/len(aggregated_df):.2%}")

In [ ]:
# Count occurrences per song number
counts = df['song_nr'].value_counts().reset_index()
counts.columns = ['song_nr', 'count']

# Merge with song names (assuming song_name is in the original df)
counts = counts.merge(df[['song_nr', 'song_name']].drop_duplicates(), on='song_nr')

# Sort by count descending
counts = counts.sort_values(by='count', ascending=False)

# Print song_name and count
print(counts[['song_name', 'count']])

In [ ]:
fin = "20250528_scopus_wordplay_table.csv"
df = pd.read_csv(fin, sep="\t")

In [ ]:
unique_songs = df['song_name'].unique()

for song in unique_songs:
    subset = df[df['song_name'] == song]
    has_yes = any(subset['wordplay'] == "Yes")
    has_no  = any(subset['wordplay'] == "No")

    if has_yes and has_no:
        row = subset[subset['wordplay'] == "Yes"][['title', 'explanation']].iloc[0]
        print(f"\nSong: {song} ---")
        print(f"Yes: {row['title']}, {row['explanation']}")

        row = subset[subset['wordplay'] == "No"][['title', 'explanation']].iloc[0]
        print(f"No: {row['title']}, {row['explanation']}")

In [ ]:
def build_user_prompt(song_title: str, titles_to_classify: list[dict]) -> str:
    """
    Builds a *plain-language* prompt that:
      • Shows six few-shot demonstrations as natural text (one per block).  
      • Separates the “Examples” section from the “Task” section with a clear delimiter.  
      • Lists the titles to classify in a numbered list that is easy for the model to read.  
    The returned string is intended to be the *content* of **one** user-turn in the chat.
    """
    # ---- FEW-SHOT DEMONSTRATIONS -------------------------------------------------
    few_shot_examples = [
        {
            "beatles_song": "Let It Be",
            "paper_title": "There will be an answer; let it bleed",
            "explanation": "quotes the lyric and twists 'be' → 'bleed'",
            "answer": "Yes"
        },
        {
            "beatles_song": "With a Little Help from My Friends",
            "paper_title": "Understanding the role of friends in adolescent health decisions",
            "explanation": "topic overlap only, no playful re-phrasing",
            "answer": "No"
        },
        {
            "beatles_song": "Everybody's Got Something to Hide Except Me and My Monkey",
            "paper_title": ("Everybody's got something to hide except me and my patented monkey: "
                            "patentability of cloned organisms."),
            "explanation": "direct reuse of the song title with ironic twist",
            "answer": "Yes"
        },
        {
            "beatles_song": "The Long and Winding Road",
            "paper_title": "Myocyte hypertrophy: The long and winding RhoA'd",
            "explanation": "puns on 'Road' → 'RhoA'd'",
            "answer": "Yes"
        },
        {
            "beatles_song": "All You Need Is Love",
            "paper_title": ("All You Need Is RAW: Defending Against Adversarial Attacks "
                            "with Camera Image Pipelines"),
            "explanation": "swaps 'Love' for 'RAW' in the phrase",
            "answer": "Yes"
        },
        {
            "beatles_song": "Come Together",
            "paper_title": ("Come for climate, stay for community: Acting, emoting, and staying "
                            "together through the climate crisis"),
            "explanation": "contains all the same words but not used as a song reference",
            "answer": "No"
        },
    ]

    # ----- Assemble example blocks as readable prose -----------------------------
    example_blocks = []
    for ex in few_shot_examples:
        block = (
            f"Example – **Song:** {ex['beatles_song']}\n"
            f"**Title:** \"{ex['paper_title']}\"\n"
            f"Assistant: {ex['answer']} – {ex['explanation']}"
        )
        example_blocks.append(block)
    examples_section = "\n\n".join(example_blocks)

    # ----- Assemble the task list ------------------------------------------------
    task_lines = []
    for idx, item in enumerate(titles_to_classify, start=1):
        scopus_id = item.get("scopus_id", "MISSING_ID")
        title = item["title"]
        task_lines.append(f"{idx}. {scopus_id} — \"{title}\"")
    task_section = "\n".join(task_lines)

    # ----- Final prompt string ---------------------------------------------------
    prompt = (
        f"### Song under consideration\n"
        f"{song_title}\n\n"
        f"### Annotated examples\n"
        f"{examples_section}\n\n"
        f"---\n"
        f"### Task\n"
        f"Classify each title below as a *humorous word-play* on the song **{song_title}** "
        f"(`Yes` or `No`) and give a one-sentence explanation.\n"
        f"Return your answers as the JSON array specified in the system instructions.\n\n"
        f"{task_section}"
    )

    return prompt

In [ ]:
# def build_system_prompt(song: str) -> str:
# 	return  f"""
# 	You are a helpful and witty research assistant specialized in language and humor recognition. Your task is to determine whether each academic paper title in the list below is a humorous wordplay on the Beatles song title {song}. 
# 	A humorous wordplay means the title creatively references or rephrases the Beatles song in a witty, ironic, or playful way—beyond merely reusing the exact song title.

# 	You should explain your reasoning in one sentence before giving your final answer. Your output must be a JSON array where each item has:
# 		•	scopus_id: the unique identifier of the paper,
# 		•	paper_title: the title of the paper,
# 		•	explanation: a one-sentence explanation of your reasoning,
# 		•	answer: either "Yes" or "No" indicating whether the title is a humorous wordplay on the song.

# 	Always prioritize concise, clear, and justified explanations. Only say "Yes" when there's clear humorous intent or play on words involving the song title.
# 	"""

In [ ]:
def build_system_prompt(song: str) -> str:
    return f"""
You are a helpful and witty research assistant specialised in language- and humour-recognition.

## Task
For every paper title you receive, decide whether it is a **humorous word-play** on the Beatles song **“{song}.”**
A title usually qualifies if it…
• Quotes the song verbatim **with one or two words swapped** (e.g. “All You Need Is Data”).  
• Uses a rhyme, pun, or near-synonym in place of the original word(s).  
Exact reuse *without* any twist, or mere topical overlap, is **not** enough.

## Output
Return one JSON array and nothing else.

```json
{{ "scopus_id": "...", "paper_title": "...",
   "explanation": "One sentence.", "answer": "Yes" | "No" }}
```
    """

In [ ]:
def parse_truncated_json(json_text: str):
    # Try parsing, and back off until successful
    lines = json_text.strip().splitlines()
    
    for i in range(len(lines), 0, -1):
        try:
            partial = "\n".join(lines[:i])
            # Try to close the JSON array if it's missing
            if not partial.strip().endswith(']'):
                partial += "\n]"
            return json.loads(partial)
        except json.JSONDecodeError:
            continue

    raise ValueError("Could not parse any valid portion of the JSON.")

In [ ]:
# Group by song_name
tok_in, tok_out = 0, 0
count = 0
for song, group in tqdm(df.groupby("song_name"), total=df["song_name"].nunique(), desc="Songs"):
    if count > -1:
        titles = group[["scopus_id", "title"]].to_dict(orient="records")
        #titles = titles[:2]  # Limit to first 10 titles for brevity
        system_prompt = build_system_prompt(song)
        user_prompt = build_user_prompt(song, titles)
        print(system_prompt + "\n" + user_prompt)

        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=TEMPERATURE
        )
        tok_in += response.usage.prompt_tokens
        tok_out += response.usage.completion_tokens
        
        # Parse the JSON string into Python objects
        raw_json = response.choices[0].message.content
        clean_json = raw_json.strip().removeprefix("```json").removesuffix("```").strip()
        results = parse_truncated_json(clean_json)  # JSON is potentially truncated

        # Iterate through and unpack
        with open(fout, "a", encoding="utf-8") as f_out:
            for entry in results:
                scopus_id = entry["scopus_id"]
                paper_title = entry["paper_title"]
                explanation = entry["explanation"]
                answer = entry["answer"]

                f_out.write(f"{scopus_id}\t{paper_title}\t{explanation}\t{answer}\n")

    count += 1

In [ ]:
# Iterate through and unpack
with open(fout, "a", encoding="utf-8") as f_out:
    for entry in results:
        scopus_id = entry["scopus_id"]
        paper_title = entry["paper_title"]
        explanation = entry["explanation"]
        answer = entry["answer"]

        f_out.write(f"{scopus_id}\t{paper_title}\t{explanation}\t{answer}\n")

In [ ]:
print(f"Input tokens: {tok_in}, Output tokens: {tok_out}")
print(f"Input cost: ${tok_in / 1_000_000 * token_costs[MODEL]['input']:.2f}, output cost: ${tok_out / 1_000_000 * token_costs[MODEL]['output']:.2f}")
print(f"Total cost: ${tok_in / 1_000_000 * token_costs[MODEL]['input'] + tok_out / 1_000_000 * token_costs[MODEL]['output']:.2f}")

### Inspect wordplay

In [ ]:
df_tags = pd.read_csv(fout, sep="\t")
df_refs = pd.read_csv(fin, sep="\t")
df_wordplay = pd.merge(df_tags[df_tags["wordplay"] == "Yes"], df_refs, on="scopus_id", how="inner") 

n_total = len(df_tags)
n_wordplay = len(df_wordplay)
print(f'{n_wordplay/n_total:.2%} of the titles were classified as humorous wordplay on a Beatles song title.')

In [ ]:
# Loop over rows in df_wordplay and print song_name, title and explanation
with open("wordplay.txt", "w", encoding="utf-8") as f:
    for _, row in df_wordplay.iterrows():
        f.write(f"Song: {row['song_name']}\nTitle: {row['title']}\nExplanation: {row['explanation']}\n\n")

In [ ]:
song_title = "The Long and Winding Road"
title = "Myocyte hypertrophy: The long and winding RhoA'd" 

print(fuzz.partial_ratio(title.lower(), song_title.lower()))

In [ ]:
### Test OpenAI API with a simple completion request
completion = client.chat.completions.create(
  model=MODEL,
  store=True,
  messages=[
    {"role": "system", "content": "You are a helpful and witty research assistant specialized in language and humor recognition. Your task is to determine whether each academic paper title in the list below is a humorous wordplay on the Beatles song title 'All You Need Is Love'. A humorous wordplay means the title creatively references or rephrases the Beatles song in a witty, ironic, or playful way—beyond merely reusing the exact song title. You should explain your reasoning in one sentence before giving your final answer."},
    {"role": "user", "content": "Is 'Moving Object Segmentation: All You Need is SAM (and Flow)' a wordplay on the Beatles song 'All You Need Is Love'?"},
  ]
)

print(completion.choices[0].message.content)

In [ ]:
few_shot_examples = [
        {
            "scopus_id": "SCOPUS_ID:84896352755",
            "paper_title": "There will be an answer; let it bleed",
            "explanation": "The title mimics two exact lines 'There will be an answer, let it be' from the song text of 'Let It Be' with a twist ('Let it bleed'), which is a clear humorous wordplay.",
            "answer": "Yes",
            "song": "Let It Be"
        },
        {
            "scopus_id": "SCOPUS_ID:XXX",
            "paper_title": "Understanding the role of friends in adolescent health decisions",
            "explanation": "Although thematically similar, the title does not use a creative or humorous twist on the Beatles song title.",
            "answer": "No",
            "song": "With a Little Help from My Friends"
        },
        {
            "scopus_id": "SCOPUS_ID:33646982325",
            "paper_title": "Everybody's got something to hide except me and my patented monkey: patentability of cloned organisms.",
            "explanation": "This is a direct reuse of the song title in the context of the paper with light irony, fitting the category of humorous wordplay.",
            "answer": "Yes",
            "song": "Everybody's Got Something to Hide Except Me and My Monkey"
        },
        {
            "scopus_id": "SCOPUS_ID:XXX",
            "paper_title": "Myocyte hypertrophy: the long and winding RhoA'd",
            "explanation": "This is a clever wordplay with the name of the song and the protein RhoA.",
            "answer": "Yes",
            "song": "The Long and Winding Road"
        },
        {
            "scopus_id": "SCOPUS_ID:85194998005",
            "paper_title": "Come for climate, stay for community: Acting, emoting, and staying together through the climate crisis",
            "explanation": "It is just a coincidence that all words in the song title appear in this paper title, it is not a reference to the song.",
            "answer": "No",
            "song": "Come Together"
        }
    ]

for example in few_shot_examples:
    print(f"Paper Title: {example['paper_title']}")
    print(f"Song: {example['song']}")
    
    fuzz_score = fuzz.partial_ratio(example['paper_title'].lower(), example['song'].lower())
    print(f"Fuzz: {fuzz_score:.3f}\n")

In [ ]:
lucy_res = ["Lucy in the sky with Trojan asteroids", 
            "Lucy in the Sky with Crystals", 
            "Lucy in the sky with ketamine: Psychoactive drugs have potential for a major breakthrough in treating depression",
            "Discourse in the sky with diamonds",
            "Leucine in the sky with diamonds",
            "Rubies in the sky with diamonds: Prices are up, thanks to extravagant Asians and edgy Westerners"]
for example in lucy_res:
    print(f"Paper Title: {example}")
    
    fuzz_score = fuzz.partial_ratio(example.lower(), "Lucy In The Sky With Diamonds".lower())
    print(f"Fuzz: {fuzz_score:.3f}\n")

In [ ]:
let_res = ["“You don’t have the right resources to let it hurt”: How structural vulnerabilities shape opioid withdrawal experiences among a community sample of people who inject drugs in Los Angeles, California", 
            "Let It Bleed", 
            "Let it shine: Autofluorescence of Papanicolaou-stain improves AI-based cytological oral cancer detection",
            ]
for example in let_res:
    print(f"Paper Title: {example}")
    
    fuzz_score = fuzz.partial_ratio(example.lower(), "Let It Be".lower())
    print(f"Fuzz: {fuzz_score:.3f}\n")

# Final Processing

In [ ]:
# fname = "beatles_songs_wordplay.txt"
# fin1 = "20250528_scopus_approx.txt"
# fin2 = "20250528_scopus_wordplay_table_full.csv"
# fin3 = "20250707_scopus_approx.txt"
# fout = "20250707_scopus_wordplay.txt"

fin1 = "20250707_scopus_wordplay.txt"
#fin1 = "20250715_scopus_lyrics_temp.txt"
#fout = "20250715_scopus_lyrics_wordplay.txt"

In [ ]:
# Load datasets
df1 = pd.read_csv(fin1, sep="\t", encoding="utf-8")
#df2 = pd.read_csv(fin2, sep=";", encoding="utf-8")
#df3 = pd.read_csv(fin3, sep="\t", encoding="utf-8")

# Filter to only include wordplay entries
keys = ["scopus_id", "song_name"]

print(len(df1))
df = df1.drop_duplicates(subset=keys, keep="first")
print(len(df))

df.head()

In [ ]:
df.to_csv(
    fout,
    sep="\t",
    encoding="utf-8",
    index=False
)

In [ ]:
# Assuming your DataFrame is called df and has a column 'song_name'
song_counts = (
    df.groupby('song_name')
      .size() # count entries per group
      .reset_index(name='count') # convert to DataFrame with count column
      .sort_values('count', ascending=False) # sort descending
      .reset_index()
)

song_counts["index"] = song_counts.index + 1

song_counts.to_csv(
    #"scopus_lyrics_wordplay_table.csv",
    "scopus_wordplay_table.csv",
    sep=";",
    encoding="utf-8",
    index=False
)